# Día 4 · Unidad 4 — Funciones (VI, opcional/bonus): `functools`

> 🎁 **Contenido opcional / bonus.** El programa de la Unidad 4 no exige `functools` — este notebook amplía lo visto si el grupo va sobrado de tiempo. Si vais justos, saltad directamente a `07_ejercicios.ipynb`.

**Objetivos de aprendizaje**
- Usar `functools.lru_cache` para cachear el resultado de una función costosa.
- Usar `functools.partial` para fijar de antemano parte de los argumentos de una función.
- Usar `functools.reduce` para reducir un iterable a un único valor.

**Duración estimada:** 30-40 minutos.

**Requisitos previos:** `01_teoria_funciones_basicas.ipynb` y, si lo has visto, `05_teoria_decoradores_bonus.ipynb`.

## 1. Qué es `functools`

Es un módulo estándar de Python con utilidades para funciones de orden superior (funciones que actúan sobre otras funciones, o que las devuelven).

## 2. `lru_cache`: cachear resultados

Cuando una función es costosa (tarda tiempo o consume recursos) y se llama repetidamente con los **mismos** argumentos, cachear el resultado evita recalcularlo. `lru_cache` usa la estrategia *Least Recently Used*: si la caché se llena, descarta primero el resultado usado hace más tiempo.

Para no depender de una conexión a internet real (nunca conviene que un notebook de clase falle porque falla la conexión), simulamos una consulta "lenta" a una base de datos de pólizas con `time.sleep()`:

In [1]:
import time


def consultar_poliza_bbdd(numero_poliza: str) -> dict:
    """Simula una consulta lenta a una base de datos de pólizas.

    En un sistema real, esto sería una llamada a una base de datos o a
    una API (ver Unidad 9). Aquí se simula con `time.sleep()` para no
    depender de una conexión real durante la clase.

    Args:
        numero_poliza: identificador de la póliza.

    Returns:
        dict: datos simulados de la póliza.
    """
    time.sleep(0.3)  # simula latencia de red/BBDD
    return {"numero_poliza": numero_poliza, "prima": 320.0, "estado": "activa"}


In [2]:
numeros_poliza = ["POL-001", "POL-002", "POL-003", "POL-001", "POL-002", "POL-001"]

inicio = time.perf_counter()
for numero in numeros_poliza:
    consultar_poliza_bbdd(numero)
fin = time.perf_counter()

print(f"Sin caché: {fin - inicio:.2f} s para {len(numeros_poliza)} consultas")


Sin caché: 1.80 s para 6 consultas


Como `"POL-001"` se repite tres veces y `"POL-002"` dos, estamos pagando el coste de la consulta "lenta" varias veces por el mismo dato. `@lru_cache` lo evita:

In [3]:
from functools import lru_cache


@lru_cache(maxsize=32)
def consultar_poliza_bbdd_con_cache(numero_poliza: str) -> dict:
    time.sleep(0.3)
    return {"numero_poliza": numero_poliza, "prima": 320.0, "estado": "activa"}


inicio = time.perf_counter()
for numero in numeros_poliza:
    consultar_poliza_bbdd_con_cache(numero)
fin = time.perf_counter()

print(f"Con caché: {fin - inicio:.2f} s para {len(numeros_poliza)} consultas")


Con caché: 0.90 s para 6 consultas


Con caché, sólo se paga la latencia simulada una vez por cada `numero_poliza` distinto (tres valores únicos: `POL-001`, `POL-002`, `POL-003`), en vez de una por cada consulta (seis en total).

> ⚠️ **Cuándo NO usar `lru_cache`**: sólo tiene sentido si la función es determinista (mismos argumentos → mismo resultado siempre) y sus argumentos son *hashables* (números, strings, tuplas — no listas ni diccionarios). Si `consultar_poliza_bbdd` devolviera datos que cambian con el tiempo (por ejemplo, el estado actual de un siniestro en curso), cachearla te daría datos obsoletos sin ningún aviso.

## 3. `partial`: fijar argumentos por adelantado

`functools.partial` crea una nueva función a partir de otra, con algunos argumentos ya fijados de antemano.

In [4]:
from functools import partial


def calcular_prima_total(numero_polizas: int, prima_unitaria: float) -> float:
    return numero_polizas * prima_unitaria


calcular_prima_total_auto = partial(calcular_prima_total, prima_unitaria=320.0)
calcular_prima_total_auto(numero_polizas=50)


16000.0

Un caso de uso algo más real: fijar el destino de `print` a la salida de error estándar, sin tener que repetir el argumento `file=` cada vez:

In [5]:
import sys

print_error = partial(print, file=sys.stderr)
print_error("Aviso: prima fuera de rango esperado")


Aviso: prima fuera de rango esperado


## 4. `reduce`: reducir un iterable a un único valor

`functools.reduce(funcion, iterable, valor_inicial)` aplica `funcion` acumulativamente: primero a los dos primeros elementos, luego al resultado y al tercero, y así sucesivamente, hasta quedarse con un único valor.

In [6]:
from functools import reduce
import operator


def producto(valores) -> float:
    """Calcula el producto de todos los valores de un iterable.

    Args:
        valores: iterable de números.

    Returns:
        float: el producto de todos los valores (1 si `valores` está vacío).
    """
    return reduce(operator.mul, valores, 1)


factores_recargo = [1.05, 1.10, 0.95]  # recargos/descuentos encadenados
print(producto(factores_recargo))


1.0972500000000003


Un caso clásico: encadenar varios factores de ajuste sobre una prima base (por ejemplo, recargo por zona, descuento por antigüedad, recargo por siniestralidad), todos aplicados en cadena:

In [7]:
prima_base = 400.0
factor_total = producto(factores_recargo)
prima_final = prima_base * factor_total
print(f"Prima final: {prima_final:.2f}")


Prima final: 438.90


**Para ti:** usa `functools.reduce` para escribir una función `prima_total_cartera(primas)` que sume una lista de primas usando `operator.add` en vez del bucle `for` con `+=` que ya conoces. Compara el resultado con `sum(primas)`: ¿coinciden?

## Resumen

- `functools.lru_cache` cachea el resultado de una función según sus argumentos; sólo tiene sentido si la función es determinista y los argumentos son *hashables* — nunca la uses sobre una función cuyo resultado cambie con el tiempo.
- `functools.partial` crea una nueva función con algunos argumentos ya fijados, útil para simplificar llamadas repetitivas con parte de la configuración siempre igual.
- `functools.reduce(funcion, iterable, inicial)` colapsa un iterable en un único valor aplicando `funcion` acumulativamente — pares como `producto`/`reduce(operator.mul, ...)` o `sum`/`reduce(operator.add, ...)`.
- Simular una operación "lenta" con `time.sleep()` (en vez de una llamada de red real) mantiene el notebook reproducible sin depender de la conexión a internet.

**Siguiente:** `07_ejercicios.ipynb`, para practicar todo el bloque de Funciones.